In [17]:
import os
import time
from google import genai
from google.genai import types
from dotenv import load_dotenv
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


from datasets import load_dataset

In [18]:
ds = load_dataset("cardiffnlp/tweet_eval", "sentiment")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12284 entries, 0 to 12283
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    12284 non-null  object
 1   label   12284 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 192.1+ KB


In [19]:
test['label'] = test['label'].apply(lambda x: 'negative' if x == 0 else 'neutral' if x == 1 else 'positive')

labels = test['label'].unique()

test

,text,label
0,@user @user what do these '1/2 naked pics' hav...,neutral
1,OH: “I had a blue penis while I was this” [pla...,neutral
2,"@user @user That's coming, but I think the vic...",neutral
3,I think I may be finally in with the in crowd ...,positive
4,"@user Wow,first Hugo Chavez and now Fidel Cast...",negative
...,...,...
12279,Sentinel Editorial: FBI’s Comey ‘had no one of...,neutral
12280,perfect pussy clips #vanessa hudgens zac efron...,neutral
12281,#latestnews 4 #newmexico #politics + #nativeam...,neutral
12282,Trying to have a conversation with my dad abou...,negative


In [20]:
load_dotenv()

api_key=os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

In [21]:
test2 = pd.read_csv('results/partial_gemini_ZS_multiclass1_2.csv')

In [22]:
test2 = test2[test2['prediction'].isnull()]
test = test2[['text', 'label']]

In [23]:
def classify(text, labels):

    sys_instruct="You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multiclass classification tasks based on user instructions."

    start_time = time.time()

    response = client.models.generate_content(
        model="gemini-2.0-flash",
        config=types.GenerateContentConfig(
            system_instruction=sys_instruct,
            safety_settings=[
            types.SafetySetting(
                category="HARM_CATEGORY_HARASSMENT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_HATE_SPEECH",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_DANGEROUS_CONTENT",
                threshold="BLOCK_NONE"
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_CIVIC_INTEGRITY",
                threshold="BLOCK_NONE"
            ),
            ],
        ),
        contents=f"Classify the following text based on the task: Sentiment analysis of tweets. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"
    )

    request_time = time.time() - start_time
    completion = response.text
    if completion:
        completion = completion.lower()
    else:
        completion = "None"
    completion_tokens = response.usage_metadata.candidates_token_count
    prompt_tokens = response.usage_metadata.prompt_token_count
    total_tokens = response.usage_metadata.total_token_count

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    if 'positive' in text:
        return 'positive'
    elif 'negative' in text:
        return 'negative'
    elif 'neutral' in text:
        return 'neutral'
    else:
        return 'error'

In [24]:
pred_df = test.copy() 

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_gemini_ZS_multiclass1_1.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/gemini_ZS_multiclass1.csv", index=False)

pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
5676,Melania Trump Gives Rare Campaign Speech,neutral,neutral\n,0.716638,2.0,85.0,87.0,neutral
5677,New picture from #BeautyAndTheBeast #DisneyLiv...,positive,positive\n,1.147945,2.0,95.0,97.0,positive
5678,@user Putin's endgame is to get states to diss...,negative,negative\n,1.078744,2.0,105.0,107.0,negative
5679,.@DailyMailUK front page | Swear oath to live ...,negative,neutral\n,1.306177,2.0,103.0,105.0,neutral
5680,PROOF HILLARY WILL CHOOSE DEATH OVER PEACE #li...,negative,negative\n,1.112323,2.0,95.0,97.0,negative
...,...,...,...,...,...,...,...,...
12279,Sentinel Editorial: FBI’s Comey ‘had no one of...,neutral,neutral\n,1.156851,2.0,94.0,96.0,neutral
12280,perfect pussy clips #vanessa hudgens zac efron...,neutral,negative\n,1.087179,2.0,89.0,91.0,negative
12281,#latestnews 4 #newmexico #politics + #nativeam...,neutral,neutral\n,1.024140,2.0,109.0,111.0,neutral
12282,Trying to have a conversation with my dad abou...,negative,negative\n,1.147010,2.0,98.0,100.0,negative


In [48]:
y_pred = pred_df['prediction_post_processed']
y_true = pred_df['label']

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.675431
F1 score: 0.666851
Precision: 0.700806
Recall: 0.675431


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [49]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 0.9522193817003897
Average completion tokens: 2.0
Average prompt tokens: 99.66403451644416
Average total tokens: 101.66354607619668


In [56]:
input_token_price = 0.1/1_000_000
output_token_price = 0.4/1_000_000

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    # if cost is null print row
    if pd.isnull(cost):
        cost = 0
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.13222379999999992


In [57]:
with open('results/gemini_ZS_multiclass1.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')